# New axes for a cloud of points

What does it mean to draw a new axis through a cloud of points?

Take the vocabulary and reading scores from the previous notebook and subtract each mean. The four centred points are $(-3, -3)$, $(-1, -1)$, $(1, 1)$, and $(3, 3)$. They lie on a straight line.

One new axis can be drawn through that cloud. The closest-fit line, for a line that passes through the centre, is the same direction as the axis that collects the most spread. Each centred test has a sum of squares of 20. Together the two tests have 40. The axis along the points collects all 40. An axis at right angles to the points collects none.

These four points were built to lie on a line. A cloud of questionnaire scores is wider, and a second axis then carries some of the spread. The picture uses two tests only.

Teaching note. Pearson, K. (1901). On lines and planes of closest fit to systems of points in space. *The London, Edinburgh, and Dublin Philosophical Magazine and Journal of Science, 2*(11), 559-572. https://doi.org/10.1080/14786440109462720


## The fitted direction

The cell draws the centred points and finds the direction with a singular value decomposition. The first time a cell imports the plotting library, that library downloads. NumPy comes with it.


In [ ]:
# Four centred points on a line, and the direction a singular value decomposition finds through them.
import numpy as np
import matplotlib.pyplot as plt

# Vocabulary and reading, already centred. The lists match, so each point lies on y = x.
vocab_c = [-3, -1, 1, 3]
reading_c = [-3, -1, 1, 3]

# Sum of squares of one centred test.
ss_one = 0
for score in vocab_c:
    ss_one = ss_one + score ** 2
# Both tests together.
ss_both = ss_one + ss_one
print(ss_one)
print(ss_both)

# One row per student.
points = np.column_stack([vocab_c, reading_c])
# The first row of vt is the direction. s holds the spread along each direction.
u, s, vt = np.linalg.svd(points, full_matrices=False)
direction = vt[0].copy()
# Point the direction towards the higher vocabulary scores.
if direction[0] < 0:
    direction = -direction
print([round(float(value), 3) for value in direction])
# Spread along the fitted direction, then along the other direction.
print(round(float(s[0] ** 2), 1))
print(round(float(s[1] ** 2), 1))

# The cloud, with the fitted direction drawn through the origin.
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.scatter(vocab_c, reading_c)
span = 4.2
ax.plot(
    [-span * float(direction[0]), span * float(direction[0])],
    [-span * float(direction[1]), span * float(direction[1])],
)
ax.axhline(0, color="0.85", linewidth=0.8)
ax.axvline(0, color="0.85", linewidth=0.8)
ax.set_aspect("equal")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-4.5, 4.5)
ax.set_xlabel("Vocabulary, centred")
ax.set_ylabel("Reading, centred")
ax.set_title("The fitted axis")
plt.show()


## Turn the axis

The player turns one axis from horizontal to 180 degrees. Each frame draws the projections onto that axis and reports the sum of their squares. The sum is largest along the points, at 45 degrees, and it falls to zero at 135 degrees.

Run the cell, then press play under the figure, or drag the slider. The player is HTML in the page. Redrawing the frames can print a repeated warning from this plotting build. The cell hides that warning so the player stays easy to see.


In [ ]:
# Turn one axis and watch the sum of squared projections. The points are the same centred scores.
import math
import warnings
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Hide a warning from this version of Matplotlib, so the player can start cleanly. The warning is about an old way of passing coordinates, and it does not change the figure.
warnings.filterwarnings("ignore", message="The x parameter as float was deprecated")
warnings.filterwarnings("ignore", message="The y parameter as float was deprecated")

# The same centred scores as the cell above.
vocab_c = [-3, -1, 1, 3]
reading_c = [-3, -1, 1, 3]

def projection_ss(degrees):
    # A unit axis at this angle. The projection of a point is the dot product with that axis.
    theta = math.radians(degrees)
    ux = math.cos(theta)
    uy = math.sin(theta)
    total = 0
    for i in range(4):
        along = vocab_c[i] * ux + reading_c[i] * uy
        total = total + along ** 2
    return total

# Horizontal, along the points, and at right angles to the points.
ss_horizontal = projection_ss(0)
ss_diagonal = projection_ss(45)
ss_across = projection_ss(135)
print(round(ss_horizontal, 1))
print(round(ss_diagonal, 1))
print(round(ss_across, 1))

# One frame every 15 degrees, from 0 through 180.
angles = []
for step in range(13):
    angles.append(15 * step)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-4.5, 4.5)
ax.set_aspect("equal")
ax.axhline(0, color="0.85", linewidth=0.8)
ax.axvline(0, color="0.85", linewidth=0.8)
ax.scatter(vocab_c, reading_c, zorder=3)
axis_line, = ax.plot([], [], color="C0")
# One grey segment per point, from the point to its projection.
proj_lines = []
for i in range(4):
    segment, = ax.plot([], [], color="0.55", linewidth=0.8)
    proj_lines.append(segment)
ax.set_xlabel("Vocabulary, centred")
ax.set_ylabel("Reading, centred")

def draw(frame):
    # The unit axis for this frame.
    theta = math.radians(angles[frame])
    ux = math.cos(theta)
    uy = math.sin(theta)
    axis_line.set_data([-4.2 * ux, 4.2 * ux], [-4.2 * uy, 4.2 * uy])
    total = 0
    for i in range(4):
        along = vocab_c[i] * ux + reading_c[i] * uy
        total = total + along ** 2
        proj_lines[i].set_data(
            [vocab_c[i], along * ux],
            [reading_c[i], along * uy],
        )
    ax.set_title(
        "Angle " + str(angles[frame]) + " degrees: sum of squares " + str(round(total, 1))
    )

# Build a player under the figure. Press play, or drag the slider, to turn the axis.
player = FuncAnimation(fig, draw, frames=len(angles), interval=500, blit=False, repeat=False)
page = player.to_jshtml()
# Close the static figure so it is not also shown above the player.
plt.close(fig)
HTML(page)


## Check yourself

1. The sum of squares of the centred vocabulary scores `[-3, -1, 1, 3]`. This is also the sum of squared projections onto the horizontal axis.
2. The sum of squared projections onto the axis at 45 degrees. For these points, that sum is twice the sum in question 1.

Type your answer in place of `...` in the cell below, then run the cell. A match prints `That matches.` A first miss prints a hint. Run the cell again, or pass `show=True`, to see the expected result. The cell loads `check` from `check.py` in the file list.

If the blank is inside a function, include `return` so the function gives a value back.


In [ ]:
# Load check from check.py so this cell can compare what you type with the stored result.
# On the notebook page that file is at /drive. Opened from a lesson folder, it is in the parent folder.
import sys
from pathlib import Path

# Use /drive when the file is there. Otherwise use the parent folder.
if Path("/drive/check.py").is_file():
    root = "/drive"
elif (Path.cwd().parent / "check.py").is_file():
    root = str(Path.cwd().parent)
else:
    root = ""
# Add that folder to the places Python searches, then load check.
if root and root not in sys.path:
    sys.path.append(root)
from check import check
# Question 1: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("pca02-1", answer, hint="Square each centred score and add.")
# Question 2: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("pca02-2", answer, hint="Along these points the sum is twice the sum of squares of one centred test.")


## What you practised

You centred two tests, found the direction through the points, and turned an axis until the sum of squared projections was as large as the two tests allowed. The points were built to lie on a line, so the second direction had no spread left.
